# 00 – Setup, uji lingkungan, dan benchmark (CP0) — Versi 2 (notebook mandiri)

Notebook ini **tidak mengimpor folder `src`** dan **tidak memakai Docker**. Semua kode yang dibutuhkan ada di sel-sel di bawah.

Isi:
1. Pemasangan pustaka (sekali saja) dan pemuatan konfigurasi `configs/base.yaml`.
2. Uji lingkungan: pustaka, GPU, dataset, bobot Swin-T.
3. Benchmark Swin-T (tensor acak, bukan data nyata).
4. Benchmark GAN pada 8 GB VRAM: menyapu `micro_batch` (gradient accumulation) dan mengukur **puncak memori termasuk langkah R1**.
5. Estimasi jam total dan rekomendasi `gan.micro_batch` serta `gan.total_kimg`.

**Kejujuran hasil:** semua angka benchmark memakai tensor acak. Notebook ini hanya mengukur kecepatan dan memori, bukan kualitas.
Hasil disimpan di `outputs/cp0/`. Nilai konfigurasi **tidak diubah otomatis**: ubah `configs/base.yaml` dengan tangan dan catat di `LOG.md` (RULES §4).

Perbedaan dengan Versi 1: dataset tidak dipasang `:ro` (Windows), jadi baca-saja hanya disiplin; tidak ada proses terpisah per job, sehingga memori GPU dibersihkan manual.

## 1. Pemasangan pustaka (opsional, sekali saja)

Ubah `INSTALL = True` hanya pada jalan pertama, di dalam venv/conda yang baru. Torch memakai indeks CUDA 12.8 (RTX 3070 Ti). Pustaka untuk notebook 03 ke atas (PyRadiomics, ctgan, lpips, dst.) dipasang di notebook tersebut.

In [ ]:
INSTALL = False

if INSTALL:
    import subprocess, sys
    pip = [sys.executable, "-m", "pip", "install"]
    subprocess.check_call(pip + ["torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu128"])
    subprocess.check_call(pip + ["transformers", "numpy", "pandas", "scipy", "pyyaml", "psutil", "matplotlib", "tqdm", "ipykernel"])

## 2. Impor, konfigurasi, dan utilitas (diambil dari `config.py` dan `utils.py` Versi 1)

In [ ]:
from __future__ import annotations

import contextlib, copy, datetime as dt, gc, hashlib, importlib, io, json, math, os, platform, random, re, shutil, subprocess, sys, tempfile, time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import yaml
from IPython.display import display, Image

# expandable_segments mengurangi fragmentasi memori, tetapi belum didukung di Windows.
if platform.system() != "Windows":
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import torch.nn as nn
import torch.nn.functional as F


def find_root() -> Path:
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "configs" / "base.yaml").exists():
            return q
    raise FileNotFoundError("configs/base.yaml tidak ditemukan dari direktori kerja ke atas")


ROOT = find_root()
os.chdir(ROOT)

_ENV = {"LARYNX_DATASET": "dataset", "LARYNX_OUTPUTS": "outputs", "LARYNX_HF_CACHE": "hf_cache"}


def deep_merge(a: dict, b: dict) -> dict:
    out = copy.deepcopy(a)
    for k, v in b.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else copy.deepcopy(v)
    return out


def _load(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    base = cfg.pop("_base", None)
    return deep_merge(_load(path.parent / base), cfg) if base else cfg


def load_merged(path) -> dict:
    cfg = _load(Path(path))
    for env, key in _ENV.items():
        if os.environ.get(env):
            cfg["paths"][key] = os.environ[env]
    for key, val in cfg["paths"].items():           # jalur relatif -> mutlak terhadap folder Versi 2
        if not Path(val).is_absolute():
            cfg["paths"][key] = str((ROOT / val).resolve())
    return cfg


def P(cfg: dict, *parts: str) -> Path:
    return Path(cfg["paths"]["outputs"]).joinpath(*parts)


def config_hash(cfg: dict) -> str:
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")).hexdigest()


def job_seed(base_seed: int, *parts) -> int:
    key = "|".join([str(base_seed), *map(str, parts)]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:4], "big")


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def atomic_write_bytes(path, data: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


def atomic_write_json(path, obj) -> None:
    atomic_write_bytes(path, json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8"))


def atomic_torch_save(obj, path) -> None:
    buf = io.BytesIO(); torch.save(obj, buf); atomic_write_bytes(path, buf.getvalue())


def is_done(job_dir, cfg_hash: str) -> bool:
    done = Path(job_dir) / "DONE"
    return done.exists() and done.read_text(encoding="utf-8").strip() == cfg_hash


def mark_done(job_dir, cfg_hash: str) -> None:
    atomic_write_bytes(Path(job_dir) / "DONE", cfg_hash.encode("utf-8"))


def git_commit() -> str:
    # Versi 2 tanpa Docker: hash commit dari git (bila ada), ditambah -dirty bila ada perubahan belum di-commit.
    try:
        c = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=15).stdout.strip()
        d = subprocess.run(["git", "-C", str(ROOT), "status", "--porcelain"], capture_output=True, text=True, timeout=15).stdout.strip()
        return (c or "tanpa-commit") + ("-dirty" if d and c else "")
    except Exception:  # noqa: BLE001
        return "tanpa-commit"


GIT_COMMIT = git_commit()
CFG_PATH = "configs/base.yaml"
cfg = load_merged(CFG_PATH)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = P(cfg, "cp0")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT   :", ROOT)
print("dataset:", cfg["paths"]["dataset"])
print("outputs:", cfg["paths"]["outputs"])
print("device :", DEV, "| commit:", GIT_COMMIT, "| config_hash:", config_hash(cfg)[:12])

def free_gpu() -> None:
    # Pengganti 'proses baru per job' di Versi 1: dipanggil di akhir tiap job agar memori GPU kembali bersih.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


WORK_DIR = Path(cfg["paths"]["outputs"]) / "_workers"


def install_module(name: str, src: str):
    # Windows memakai 'spawn': fungsi/kelas yang dipakai proses anak harus berada di modul yang dapat diimpor. Modul dibuat
    # saat notebook berjalan (satu-satunya sumber kodenya ada di sel notebook ini) lalu diimpor, termasuk oleh proses anak.
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    p = WORK_DIR / f"{name}.py"
    changed = (not p.exists()) or p.read_text(encoding="utf-8") != src
    if changed:
        p.write_text(src, encoding="utf-8")
    if str(WORK_DIR) not in sys.path:
        sys.path.insert(0, str(WORK_DIR))
    os.environ["PYTHONPATH"] = str(WORK_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
    importlib.invalidate_caches()
    mod = importlib.import_module(name)
    return importlib.reload(mod) if changed and name in sys.modules else mod

## 3. Utilitas perangkat dan Swin-T (dari `models.py` Versi 1)

In [ ]:
def configure_threads(cfg: dict) -> int:
    n = (cfg.get("compute") or {}).get("torch_threads") or os.cpu_count() or 1
    torch.set_num_threads(int(n))
    return int(n)


def use_amp(cfg: dict) -> bool:
    return bool((cfg.get("compute") or {}).get("amp", False)) and DEV == "cuda"


def autocast(cfg: dict):
    return torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp(cfg) else contextlib.nullcontext()


def build_classifier(cfg: dict, pretrained: bool | None = None):
    from transformers import AutoConfig, AutoModelForImageClassification

    m = cfg["model"]
    pretrained = m["pretrained"] if pretrained is None else pretrained
    if pretrained:
        return AutoModelForImageClassification.from_pretrained(
            m["hf_name"], cache_dir=cfg["paths"]["hf_cache"], num_labels=m["num_classes"], ignore_mismatched_sizes=True)
    conf = AutoConfig.from_pretrained(m["hf_name"], cache_dir=cfg["paths"]["hf_cache"], num_labels=m["num_classes"])
    return AutoModelForImageClassification.from_config(conf)


def build_encoder(cfg: dict):
    from transformers import SwinModel

    return SwinModel.from_pretrained(cfg["model"]["hf_name"], cache_dir=cfg["paths"]["hf_cache"])


def num_workers(cfg: dict) -> int:
    v = cfg["loader"]["num_workers"]
    if v == "auto":
        return 0 if DEV == "cpu" else max(0, min(6, (os.cpu_count() or 2) - 1))
    return int(v)


def make_loader(ds, cfg: dict, batch_size: int, workers=None, **kw):
    # workers=0 untuk pembacaan evaluasi/embedding (tanpa augmentasi, murah): menghindari spawn proses berulang di Windows.
    from torch.utils.data import DataLoader

    nw = num_workers(cfg) if workers is None else int(workers)
    return DataLoader(ds, batch_size=batch_size, num_workers=nw, pin_memory=DEV == "cuda", persistent_workers=nw > 0, **kw)


@torch.no_grad()
def predict_probs(model, ds, batch_size: int, device: str, cfg: dict) -> np.ndarray:
    # Probabilitas kelas ganas (indeks 1) untuk tiap citra pada ds, urutan sama dengan ds.
    model.eval()
    out = []
    for x, _, _ in make_loader(ds, cfg, batch_size, workers=0, shuffle=False):
        with autocast(cfg):
            logits = model(pixel_values=x.to(device, non_blocking=True)).logits
        out.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)


@torch.no_grad()
def embed_iter(encoder, ds, batch_size: int, device: str, cfg: dict):
    encoder.eval()
    for x, _, j in make_loader(ds, cfg, batch_size, workers=0, shuffle=False):
        with autocast(cfg):
            e = encoder(pixel_values=x.to(device, non_blocking=True)).pooler_output
        yield j.numpy(), e.float().cpu().numpy()

## 4. Arsitektur GAN (dari `gan/networks.py`, `gan/vesselness.py`, `gan/augment.py` Versi 1)

Perubahan Versi 2: `build_nets` memakai `gan.droi_d_channels` (D_roi setengah lebar D). Notebook 04 dan 05 akan memuat salinan sel ini lagi, karena setiap notebook mandiri.

In [ ]:
# ---------------------------------------------------------------- jaringan
def _resolutions(resolution: int) -> list:
    r, out = 4, []
    while r <= resolution:
        out.append(r); r *= 2
    return out


class ModConv(nn.Module):
    def __init__(self, cin, cout, w_dim, k=3, demod=True, up=False):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(cout, cin, k, k) / math.sqrt(cin * k * k))
        self.affine = nn.Linear(w_dim, cin)
        nn.init.constant_(self.affine.bias, 1.0)
        self.k, self.demod, self.up, self.cout = k, demod, up, cout

    def forward(self, x, w):
        B, C, H, W = x.shape
        s = self.affine(w)
        wt = self.weight[None] * s[:, None, :, None, None]
        if self.demod:
            wt = wt * torch.rsqrt(wt.square().sum(dim=(2, 3, 4), keepdim=True) + 1e-8)
        if self.up:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            H, W = H * 2, W * 2
        x = x.reshape(1, B * C, H, W)
        out = F.conv2d(x, wt.reshape(B * self.cout, C, self.k, self.k), padding=self.k // 2, groups=B)
        return out.reshape(B, self.cout, H, W)


class SynthLayer(nn.Module):
    def __init__(self, cin, cout, w_dim, up=False):
        super().__init__()
        self.conv = ModConv(cin, cout, w_dim, 3, True, up)
        self.noise_strength = nn.Parameter(torch.zeros(1))
        self.bias = nn.Parameter(torch.zeros(cout))

    def forward(self, x, w):
        x = self.conv(x, w)
        x = x + torch.randn(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device, dtype=x.dtype) * self.noise_strength
        return F.leaky_relu(x + self.bias.view(1, -1, 1, 1), 0.2)


class Generator(nn.Module):
    def __init__(self, z_dim, w_dim, n_classes, channels, resolution, map_layers):
        super().__init__()
        self.z_dim, self.n_classes, self.resolution = z_dim, n_classes, resolution
        self.embed = nn.Linear(n_classes, w_dim)
        layers, d = [], z_dim + w_dim
        for _ in range(map_layers):
            layers += [nn.Linear(d, w_dim), nn.LeakyReLU(0.2)]
            d = w_dim
        self.mapping = nn.Sequential(*layers)
        ch = {int(k): v for k, v in channels.items()}
        self.const = nn.Parameter(torch.randn(ch[4], 4, 4))
        self.res = _resolutions(resolution)
        self.conv0, self.conv1, self.torgb = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
        for r in self.res:
            if r > 4:
                self.conv0[str(r)] = SynthLayer(ch[r // 2], ch[r], w_dim, up=True)
            self.conv1[str(r)] = SynthLayer(ch[r], ch[r], w_dim)
            self.torgb[str(r)] = ModConv(ch[r], 3, w_dim, k=1, demod=False)

    def forward(self, z, y):
        z = z * torch.rsqrt(z.square().mean(dim=1, keepdim=True) + 1e-8)
        c = self.embed(F.one_hot(y, self.n_classes).to(z.dtype))
        c = c * torch.rsqrt(c.square().mean(dim=1, keepdim=True) + 1e-8)
        w = self.mapping(torch.cat([z, c], dim=1))
        x = self.const[None].expand(z.shape[0], -1, -1, -1)
        img = None
        for r in self.res:
            if r > 4:
                x = self.conv0[str(r)](x, w)
            x = self.conv1[str(r)](x, w)
            rgb = self.torgb[str(r)](x, w)
            img = rgb if img is None else F.interpolate(img, scale_factor=2, mode="bilinear", align_corners=False) + rgb
        return img


class DBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.c0 = nn.Conv2d(cin, cin, 3, padding=1)
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1, bias=False)

    def forward(self, x):
        y = F.avg_pool2d(F.leaky_relu(self.c1(F.leaky_relu(self.c0(x), 0.2)), 0.2), 2)
        return (self.skip(F.avg_pool2d(x, 2)) + y) / math.sqrt(2)


def minibatch_std(x, group=4):
    B, C, H, W = x.shape
    G = min(group, B)
    while B % G:
        G -= 1
    y = x.reshape(G, -1, C, H, W)
    y = y - y.mean(dim=0)
    y = (y.square().mean(dim=0) + 1e-8).sqrt().mean(dim=(1, 2, 3))
    return torch.cat([x, y.reshape(-1, 1, 1, 1).repeat(G, 1, H, W)], dim=1)


class Discriminator(nn.Module):
    def __init__(self, in_ch, n_classes, channels, resolution):
        super().__init__()
        ch = {int(k): v for k, v in channels.items()}
        self.res = _resolutions(resolution)[::-1]
        self.fromrgb = nn.Conv2d(in_ch, ch[self.res[0]], 1)
        self.blocks = nn.ModuleList([DBlock(ch[r], ch[r // 2]) for r in self.res[:-1]])
        c4 = ch[4]
        self.conv = nn.Conv2d(c4 + 1, c4, 3, padding=1)
        self.fc = nn.Linear(c4 * 16, c4)
        self.out = nn.Linear(c4, 1)
        self.embed = nn.Embedding(n_classes, c4)
        self.c4 = c4

    def forward(self, x, y):
        h = F.leaky_relu(self.fromrgb(x), 0.2)
        for b in self.blocks:
            h = b(h)
        h = F.leaky_relu(self.conv(minibatch_std(h)), 0.2)
        h = F.leaky_relu(self.fc(h.flatten(1)), 0.2)
        return self.out(h) + (h * self.embed(y)).sum(dim=1, keepdim=True) / math.sqrt(self.c4)


# ---------------------------------------------------------------- Frangi diferensiabel untuk D_roi
def _kernels(sigma, device, dtype):
    r = int(math.ceil(3 * sigma))
    x = torch.arange(-r, r + 1, device=device, dtype=dtype)
    g = torch.exp(-x ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    return g, -x / sigma ** 2 * g, (x ** 2 / sigma ** 4 - 1 / sigma ** 2) * g, r


def _sep(img, kx, ky, r):
    x = F.pad(img, (r, r, r, r), mode="reflect")
    return F.conv2d(F.conv2d(x, kx.view(1, 1, 1, -1)), ky.view(1, 1, -1, 1))


def _blur(img, sigma):
    if sigma <= 0:
        return img
    g, _, _, r = _kernels(sigma, img.device, img.dtype)
    return _sep(img, g, g, r)


def fov_mask(rgb, thr, erode):
    with torch.no_grad():
        m = (rgb.max(dim=1, keepdim=True).values > thr).float()
        if erode > 0:
            m = -F.max_pool2d(-m, 2 * erode + 1, stride=1, padding=erode)
    return m


def frangi_soft(gray, v):
    gray = _blur(gray, v["blur"])
    best = None
    for s in v["sigmas"]:
        g, g1, g2, r = _kernels(float(s), gray.device, gray.dtype)
        hxx, hyy, hxy = _sep(gray, g2, g, r) * s ** 2, _sep(gray, g, g2, r) * s ** 2, _sep(gray, g1, g1, r) * s ** 2
        tmp = torch.sqrt(((hxx - hyy) / 2) ** 2 + hxy ** 2 + 1e-12)
        l1, l2 = (hxx + hyy) / 2 + tmp, (hxx + hyy) / 2 - tmp
        small = torch.where(l1.abs() < l2.abs(), l1, l2)
        large = torch.where(l1.abs() < l2.abs(), l2, l1)
        rb = small.abs() / (large.abs() + 1e-8)
        s2 = small ** 2 + large ** 2
        resp = torch.exp(-rb ** 2 / (2 * v["beta"] ** 2)) * (1 - torch.exp(-s2 / (2 * v["c"] ** 2)))
        resp = torch.where(large > 0, resp, torch.zeros_like(resp))
        best = resp if best is None else torch.maximum(best, resp)
    return best


def roi_soft(rgb, v):
    fov = fov_mask(rgb, v["fov_thr"], v["fov_erode"])
    green = (rgb[:, 1:2] + 1) / 2
    mean = (green * fov).sum(dim=(1, 2, 3), keepdim=True) / (fov.sum(dim=(1, 2, 3), keepdim=True) + 1e-6)
    green = torch.where(fov > 0, green, mean.expand_as(green))
    resp = frangi_soft(green, v) / v["norm"]
    return torch.sigmoid((resp - v["threshold"]) / v["temperature"]) * fov


def roi_input(rgb, roi):
    return torch.cat([(rgb + 1) * roi - 1, roi * 2 - 1], dim=1)


# ---------------------------------------------------------------- augmentasi ADA-lite
def augment(x, p, color=True):
    if p <= 0:
        return x
    B, dev = x.shape[0], x.device

    def flag(shape=(B, 1, 1, 1)):
        return (torch.rand(shape, device=dev) < p).to(x.dtype)

    x = torch.where(flag().bool(), x.transpose(2, 3), x)
    fx, fy = 1 - 2 * flag((B,)), 1 - 2 * flag((B,))
    tx = (torch.rand(B, device=dev) * 2 - 1) * 0.25 * flag((B,))
    ty = (torch.rand(B, device=dev) * 2 - 1) * 0.25 * flag((B,))
    theta = torch.zeros(B, 2, 3, device=dev, dtype=x.dtype)
    theta[:, 0, 0], theta[:, 1, 1], theta[:, 0, 2], theta[:, 1, 2] = fx, fy, tx, ty
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    if color:
        x = x + torch.randn(B, 1, 1, 1, device=dev, dtype=x.dtype) * 0.2 * flag()
        mean = x.mean(dim=(1, 2, 3), keepdim=True)
        x = (x - mean) * torch.exp2(torch.randn(B, 1, 1, 1, device=dev, dtype=x.dtype) * 0.5 * flag()) + mean
    return x


def build_nets(cfg: dict, lam: float, device: str):
    g = cfg["gan"]
    G = Generator(g["z_dim"], g["w_dim"], 2, g["channels"], g["resolution"], g["map_layers"]).to(device)
    D = Discriminator(3, 2, g["d_channels"], g["resolution"]).to(device)
    droi_ch = g.get("droi_d_channels") or g["d_channels"]          # Versi 2: D_roi lebih ramping
    Droi = Discriminator(4, 2, droi_ch, g["resolution"]).to(device) if lam > 0 else None
    return G, D, Droi


_g = build_nets(cfg, 1.0, "cpu")
print("parameter G / D / D_roi:", [f"{sum(p.numel() for p in m.parameters()) / 1e6:.1f} jt" for m in _g])
del _g

## 5. Uji lingkungan (dari `env_check.py`)

Pustaka dibagi dua: **wajib sekarang** (menggagalkan uji) dan **dipakai notebook berikutnya** (hanya dilaporkan). Pemeriksaan yang tidak dapat dijalankan dilaporkan sebagai GAGAL, DILEWATI, atau TIDAK_TERSEDIA, bukan LULUS.

In [ ]:
import importlib

REQUIRED_NOW = ["torch", "torchvision", "transformers", "numpy", "pandas", "scipy", "yaml", "psutil", "matplotlib"]
LATER = ["sklearn", "imblearn", "skimage", "PIL", "openpyxl", "radiomics", "SimpleITK", "ctgan", "lpips"]
CHECK_WEIGHTS = True      # butuh internet pada jalan pertama; sesudahnya dari outputs/hf_cache


def check_imports(names):
    out = {}
    for n in names:
        try:
            out[n] = getattr(importlib.import_module(n), "__version__", "ok")
        except Exception as e:  # noqa: BLE001
            out[n] = f"GAGAL: {e!r}"
    return out


def check_device():
    cuda = torch.cuda.is_available()
    n = torch.cuda.device_count() if cuda else 0
    return {"cuda_available": cuda, "cuda_devices": n,
            "gpu_names": [torch.cuda.get_device_name(i) for i in range(n)],
            "gpu_memory_gib": [round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2) for i in range(n)],
            "torch": torch.__version__, "torch_cuda": torch.version.cuda,
            "torch_threads": torch.get_num_threads(), "cpu_count": os.cpu_count(),
            "ram_gib": round(psutil.virtual_memory().total / 2**30, 1)}


def check_gpu_smoke():
    if not torch.cuda.is_available():
        return {"status": "TIDAK_TERSEDIA", "detail": "CUDA tidak terdeteksi"}
    try:
        lin = torch.nn.Linear(256, 256).cuda()
        x = torch.randn(64, 256, device="cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            loss = lin(x).float().square().mean()
        loss.backward(); torch.cuda.synchronize()
        return {"status": "LULUS", "detail": f"maju-mundur fp16 di {torch.cuda.get_device_name(0)}"}
    except Exception as e:  # noqa: BLE001
        return {"status": "GAGAL", "detail": repr(e)}


def check_dataset(dataset_dir):
    # Tidak menulis apa pun ke dataset (Windows tidak punya mount :ro). Hanya membaca.
    p = Path(dataset_dir)
    if not p.is_dir():
        return {"status": "GAGAL", "detail": f"folder tidak ada: {dataset_dir}"}
    exts = {".jpg", ".jpeg", ".png"}      # sama dengan audit.py (notebook 01), supaya hitungan citra konsisten
    counts = {}
    for sub in ("Benign", "Malignant"):
        d = p / sub
        counts[sub] = sum(1 for f in d.rglob("*") if f.suffix.lower() in exts) if d.is_dir() else None
    xlsx = (p / "Patients_List_Updated_Final.xlsx").exists()
    ok = all(v for v in counts.values()) and xlsx
    return {"status": "LULUS" if ok else "GAGAL", "citra": counts, "daftar_pasien_xlsx": xlsx,
            "catatan": "baca-saja tidak dapat dipaksa di Windows; jangan menulis ke folder ini (RULES §2)"}


def check_weights(cfg):
    try:
        m = build_classifier(cfg, pretrained=True)
        return {"status": "LULUS", "detail": f"bobot {cfg['model']['hf_name']} termuat, {sum(p.numel() for p in m.parameters()):,} parameter"}
    except Exception as e:  # noqa: BLE001
        return {"status": "GAGAL", "detail": f"bobot tidak dapat dimuat/diunduh: {e!r}"}


env = {"python": sys.version.split()[0], "platform": platform.platform(),
       "imports_wajib": check_imports(REQUIRED_NOW), "imports_nanti": check_imports(LATER),
       "device": check_device(), "gpu_smoke": check_gpu_smoke(),
       "dataset": check_dataset(cfg["paths"]["dataset"]),
       "weights": check_weights(cfg) if CHECK_WEIGHTS else {"status": "DILEWATI"}}
env["git_commit"] = GIT_COMMIT      # pengganti digest citra Docker (RULES §5)
_freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
env["pip_freeze_sha256"] = hashlib.sha256(_freeze.encode("utf-8")).hexdigest()
atomic_write_bytes(OUT_DIR / "pip_freeze.txt", _freeze.encode("utf-8"))
gagal = [k for k, v in env["imports_wajib"].items() if str(v).startswith("GAGAL")]
gagal += [k for k in ("dataset", "weights", "gpu_smoke") if env[k]["status"] == "GAGAL"]
env["lulus"], env["gagal"] = not gagal, gagal
atomic_write_json(OUT_DIR / "env_check.json", env)
print(json.dumps(env, indent=2, ensure_ascii=False))

## 6. Benchmark Swin-T (dari `benchmark.py`)

Tensor acak 224×224, AMP bila CUDA. Ekstrapolasi satu epoch memakai `benchmark.assumed_train_images` (asumsi, bukan pengukuran) dan mengabaikan waktu baca data, jadi epoch nyata akan **≥** angka ini.

In [ ]:
def bench_swin(cfg: dict, batch_size=None, timed=None) -> dict:
    set_seed(cfg["seed"])
    tr, bm, md_ = cfg["train"], cfg["benchmark"], cfg["model"]
    bs, n_timed = batch_size or tr["batch_size"], timed or bm["timed_steps"]
    configure_threads(cfg)
    try:
        model, pretrained = build_classifier(cfg, pretrained=True), True
    except Exception as e:  # noqa: BLE001
        print("[peringatan] bobot pralatih tidak dapat dimuat, memakai bobot acak (waktu langkah setara):", repr(e)[:120])
        model, pretrained = build_classifier(cfg, pretrained=False), False
    model = model.to(DEV).train()
    opt = torch.optim.AdamW(model.parameters(), lr=tr["lr"], weight_decay=tr["weight_decay"])
    amp = use_amp(cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    x = torch.randn(bs, 3, md_["input_size"], md_["input_size"], device=DEV)
    y = torch.randint(0, md_["num_classes"], (bs,), device=DEV)
    lossf = torch.nn.CrossEntropyLoss()
    if DEV == "cuda":
        torch.cuda.reset_peak_memory_stats()

    def step():
        opt.zero_grad(set_to_none=True)
        with autocast(cfg):
            logits = model(pixel_values=x).logits
        loss = lossf(logits.float(), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()

    for _ in range(bm["warmup_steps"]):
        step()
    if DEV == "cuda":
        torch.cuda.synchronize()
    times = []
    for _ in range(n_timed):
        t0 = time.perf_counter(); step()
        if DEV == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    mean = sum(times) / len(times)
    steps = -(-bm["assumed_train_images"] // bs)
    res = {"device": DEV, "pretrained_weights": pretrained, "data": "tensor acak (BUKAN data nyata)", "batch_size": bs,
           "timed_steps": n_timed, "mean_step_seconds": mean, "images_per_second": bs / mean,
           "assumed_train_images": bm["assumed_train_images"],
           "estimated_epoch_minutes": mean * steps / 60,
           "estimated_30_epoch_hours_per_job": mean * steps * tr["max_epochs"] / 3600,
           "amp": amp, "gpu": torch.cuda.get_device_name(0) if DEV == "cuda" else None,
           "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1e9 if DEV == "cuda" else None,
           "ram_rss_gb": psutil.Process().memory_info().rss / 1e9, "config_hash": config_hash(cfg)}
    del model, opt, x, y
    free_gpu()
    return res


swin = bench_swin(cfg)
atomic_write_json(OUT_DIR / "benchmark.json", swin)
for k, v in swin.items():
    print(f"{k}: {v}")

## 7. Benchmark GAN pada 8 GB: sapuan `micro_batch`

Pembeda dari `bench_gan.py` Versi 1:
- **Gradient accumulation:** batch efektif tetap `gan.batch_size` (32); GPU memproses `micro_batch` citra per potongan, gradien dijumlahkan, satu `optimizer.step()` per iterasi.
- **Langkah R1 ikut diukur.** Versi 1 tidak mengukurnya, padahal turunan kedua (`create_graph=True`) adalah puncak memori. R1 dijalankan **satu iterasi tambahan** dan waktunya dibagi `r1_interval`.
- Setiap kombinasi (λ, micro_batch) yang kehabisan memori dicatat sebagai `OOM`, bukan menghentikan notebook.

Data acak, hanya kecepatan dan memori. Sel ini butuh CUDA.

In [ ]:
def gan_bench(cfg: dict, lam: float, micro: int, iters: int = 6, warmup: int = 3, p: float = 0.2) -> dict:
    g, v = cfg["gan"], cfg["gan"]["vesselness"]
    B, res = g["batch_size"], g["resolution"]
    assert B % micro == 0, f"batch_size {B} harus kelipatan micro_batch {micro}"
    n_acc, gam = B // micro, g["r1_gamma"] / 2 * g["r1_interval"]
    set_seed(cfg["seed"])
    G, D, Droi = build_nets(cfg, lam, DEV)
    optG = torch.optim.Adam(G.parameters(), lr=g["lr_g"], betas=tuple(g["betas"]))
    optD = torch.optim.Adam(list(D.parameters()) + (list(Droi.parameters()) if Droi is not None else []),
                            lr=g["lr_d"], betas=tuple(g["betas"]))
    real = torch.rand(B, 3, res, res, device=DEV) * 2 - 1
    y = torch.randint(2, (B,), device=DEV)
    sp = F.softplus

    def one_iter(do_r1: bool) -> None:
        # ---- langkah D
        G.requires_grad_(False); D.requires_grad_(True)
        if Droi is not None:
            Droi.requires_grad_(True)
        optD.zero_grad(set_to_none=True)
        for s in range(0, B, micro):
            xr, yy = real[s:s + micro], y[s:s + micro]
            with torch.no_grad():
                fake = G(torch.randn(micro, G.z_dim, device=DEV), yy)
            loss = sp(D(augment(fake, p), yy)).mean() + sp(-D(augment(xr, p), yy)).mean()
            rr = None
            if Droi is not None:
                with torch.no_grad():
                    rr, rf = roi_input(xr, roi_soft(xr, v)), roi_input(fake, roi_soft(fake, v))
                loss = loss + sp(Droi(augment(rf, p, color=False), yy)).mean() + sp(-Droi(augment(rr, p, color=False), yy)).mean()
            (loss / n_acc).backward()
            if do_r1:
                ra = augment(xr, p).detach().requires_grad_(True)
                gr, = torch.autograd.grad(D(ra, yy).sum(), ra, create_graph=True)
                r1 = gr.square().sum(dim=(1, 2, 3)).mean() * gam
                if Droi is not None:
                    xa = augment(rr, p, color=False).detach().requires_grad_(True)
                    gr2, = torch.autograd.grad(Droi(xa, yy).sum(), xa, create_graph=True)
                    r1 = r1 + gr2.square().sum(dim=(1, 2, 3)).mean() * gam
                (r1 / n_acc).backward()
        optD.step()
        # ---- langkah G
        G.requires_grad_(True); D.requires_grad_(False)
        if Droi is not None:
            Droi.requires_grad_(False)
        optG.zero_grad(set_to_none=True)
        for s in range(0, B, micro):
            yy = y[s:s + micro]
            fake = G(torch.randn(micro, G.z_dim, device=DEV), yy)
            lg = sp(-D(augment(fake, p), yy)).mean()
            if Droi is not None:
                lg = lg + lam * sp(-Droi(augment(roi_input(fake, roi_soft(fake, v)), p, color=False), yy)).mean()
            (lg / n_acc).backward()
        optG.step()

    torch.cuda.reset_peak_memory_stats()
    times = []
    for i in range(warmup + iters):
        t0 = time.perf_counter(); one_iter(False); torch.cuda.synchronize()
        if i >= warmup:
            times.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); one_iter(True); torch.cuda.synchronize()
    r1_seconds = time.perf_counter() - t0
    mean = sum(times) / len(times)
    per_iter = mean + max(0.0, r1_seconds - mean) / g["r1_interval"]      # R1 lazy: tambahan biaya dibagi interval
    return {"status": "OK", "lambda_roi": lam, "micro_batch": micro, "mean_iter_seconds": mean, "r1_iter_seconds": r1_seconds,
            "seconds_per_kimg": per_iter * 1000 / B, "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1e9}


MICRO_GRID = [32, 16, 8, 4]
LAMBDAS = [0.0, 1.0]          # 0.0 = S5 (tanpa D_roi), 1.0 = biaya D_roi untuk S6
rows = []
if DEV != "cuda":
    print("CUDA tidak tersedia: benchmark GAN DILEWATI (bukan lulus).")
else:
    B_eff = cfg["gan"]["batch_size"]
    for lam in LAMBDAS:
        for mb in MICRO_GRID:
            if B_eff % mb:
                continue
            try:
                r = gan_bench(cfg, lam, mb)
            except torch.cuda.OutOfMemoryError:
                r = {"status": "OOM", "lambda_roi": lam, "micro_batch": mb}
            rows.append(r)
            free_gpu()
            print(r, flush=True)
gan_df = pd.DataFrame(rows)
gan_df

## 8. Estimasi jam total dan rekomendasi

- **Rekomendasi `micro_batch`:** yang **terbesar** yang puncak memorinya ≤ (VRAM total − `HEADROOM_GIB`). Terbesar dipilih karena paling cepat; hasil ilmiah sama untuk semua `micro_batch` (batch efektif tetap).
- **Jumlah GAN:** `n_splits × n_repeats` per skenario (S5 dan S6), ditambah pilot λ (`len(pilot.lambdas)` GAN D_roi pada satu lipatan, D2).
- Estimasi **tidak** termasuk evaluasi FID per snapshot, pembangkitan citra sintetis, dan pelatihan Swin-T.

In [ ]:
HEADROOM_GIB = 1.5        # tampilan Windows (~0,8-1 GB) + konteks CUDA + fragmentasi
rec = {}
if len(gan_df):
    total_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
    budget_gb = (total_gib - HEADROOM_GIB) * 2**30 / 1e9        # peak_gpu_gb memakai 1e9
    ok = gan_df[(gan_df.status == "OK") & (gan_df.peak_gpu_gb <= budget_gb)]
    print(f"VRAM total {total_gib:.2f} GiB; anggaran puncak {budget_gb:.2f} GB (sisa {HEADROOM_GIB} GiB)")
    for lam in LAMBDAS:
        sub = ok[ok.lambda_roi == lam]
        rec[lam] = None if sub.empty else sub.sort_values("micro_batch", ascending=False).iloc[0].to_dict()
    cfg_s = cfg["splits"]
    n_gan = cfg_s["n_splits"] * cfg_s["n_repeats"]
    n_pilot = len(cfg["pilot"]["lambdas"])
    tk = cfg["gan"]["total_kimg"]
    if rec.get(0.0) and rec.get(1.0):
        h0 = rec[0.0]["seconds_per_kimg"] * tk / 3600
        h1 = rec[1.0]["seconds_per_kimg"] * tk / 3600
        est = {"total_kimg": tk, "jam_per_GAN_S5": h0, "jam_per_GAN_S6": h1, "n_GAN_per_skenario": n_gan, "n_GAN_pilot": n_pilot,
               "jam_total_GAN": n_gan * h0 + n_gan * h1 + n_pilot * h1,
               "micro_batch_S5": rec[0.0]["micro_batch"], "micro_batch_S6": rec[1.0]["micro_batch"]}
        print(json.dumps(est, indent=2))
    else:
        est = None
        print("Tidak ada micro_batch yang muat dalam anggaran: turunkan MICRO_GRID ke 2 atau 1, atau kecilkan d_channels.")
    s_per_job = swin["estimated_30_epoch_hours_per_job"]
    print(f"\nSwin-T: batas atas {s_per_job:.2f} jam per job (30 epoch, tanpa henti dini, tanpa citra sintetis); "
          f"S1 = {cfg_s['n_splits'] * cfg_s['n_repeats']} job.")
    atomic_write_json(OUT_DIR / "benchmark_gan.json", {
        "gpu": torch.cuda.get_device_name(0), "data": "tensor acak (BUKAN data nyata)", "batch_efektif": cfg["gan"]["batch_size"],
        "sapuan": rows, "rekomendasi": {str(k): v for k, v in rec.items()}, "estimasi": est,
        "catatan": "puncak memori termasuk langkah R1; estimasi tanpa FID, pembangkitan sintetis, dan Swin-T"})

## 9. Langkah manual setelah notebook ini

1. Ubah `configs/base.yaml` dengan tangan: `gan.micro_batch` ← nilai rekomendasi, `gan.total_kimg` ← angka yang sesuai tenggat dari `jam_per_GAN_*`.
2. Tulis keduanya di `LOG.md` (tanggal, nilai, alasan) **sebelum** menjalankan notebook 04. Perubahan konfigurasi mengubah hash, jadi `last.pt` lama otomatis diabaikan.
3. Jika semua `micro_batch` melebihi anggaran memori: kecilkan `gan.d_channels`/`droi_d_channels` atau turunkan `gan.resolution`, lalu ulangi sel 7 dan catat.
4. Lanjut ke `01_audit.ipynb`.

## 10. Pemeriksaan checkpoint CP0 dan `manifest.json` (sel terakhir)

Keputusan D1 sampai D4 dikonfirmasi manusia lewat sel di bawah (`approve_decisions`), bukan otomatis. Tinjauan orang kedua lewat `approve_review`.

In [ ]:
# ---------------------------------------------------------------- checkpoint, persetujuan manusia, LOG.md
# Pengganti checkpoint.py dan approve.py Versi 1. Persetujuan MANUSIA tidak pernah dipanggil otomatis oleh notebook.
LULUS, GAGAL, MENUNGGU, TIDAK_TERSEDIA = "LULUS", "GAGAL", "MENUNGGU", "TIDAK_TERSEDIA"
DEC_PATH, BASE_PATH, LOG_PATH = ROOT / "configs" / "decisions.yaml", ROOT / "configs" / "base.yaml", ROOT / "LOG.md"


def file_sha256(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_manifest(out_dir, files, extra=None) -> Path:
    out_dir = Path(out_dir)
    entries = {}
    for f in files:
        p = Path(f)
        if not p.exists():
            raise FileNotFoundError(f"Keluaran checkpoint tidak ditemukan: {p}")
        key = str(p.relative_to(out_dir)) if p.is_relative_to(out_dir) else str(p)
        entries[key] = {"sha256": file_sha256(p), "bytes": p.stat().st_size}
    mp = out_dir / "manifest.json"
    atomic_write_json(mp, {"files": entries, **(extra or {})})
    return mp


def _item(nama: str, status: str, detail: str) -> dict:
    return {"item": nama, "status": status, "detail": detail}


def _review_item(out_dir) -> dict:
    out_dir = Path(out_dir)
    ok = (out_dir / "REVIEWED").exists()
    return _item("Tinjauan orang kedua (berkas REVIEWED)", LULUS if ok else MENUNGGU,
                 "ada" if ok else f"setelah meninjau hasil: approve_review('{out_dir.name.upper()}', 'NAMA') (RULES §3)")


def _flag_item(out_dir, name: str, nama: str, how: str) -> dict:
    ok = (Path(out_dir) / name).exists()
    return _item(nama, LULUS if ok else MENUNGGU, "ada" if ok else how)


def _decision_item(d: str) -> dict:
    with open(DEC_PATH, "r", encoding="utf-8") as f:
        dec = yaml.safe_load(f)[d]
    st = dec["status"]
    status = {"diputuskan": LULUS, "ditunda": LULUS, "menunggu_konfirmasi": MENUNGGU}.get(st, GAGAL)
    return _item(f"{d} {st}", status, str(dec.get("isi", "")))


def _finish(cp: str, out_dir, items: list, files: list, extra=None) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    blocking = [i for i in items if i["status"] in (GAGAL, MENUNGGU)]
    rep = {"checkpoint": cp, "lulus": not blocking, "items": items, "penghambat": [i["item"] for i in blocking]}
    atomic_write_json(out_dir / f"{cp.lower()}_check.json", rep)
    for i in items:
        print(f"[{i['status']:>15}] {i['item']} -- {i['detail']}")
    if rep["lulus"]:
        write_manifest(out_dir, [out_dir / f"{cp.lower()}_check.json", *[f for f in files if Path(f).exists()]],
                       extra={"checkpoint": cp, "git_commit": GIT_COMMIT, **(extra or {})})
        print(f"\n{cp} LULUS. manifest.json ditulis. Tag git '{cp.lower()}' dapat dibuat (setelah tinjauan orang kedua).")
    else:
        print(f"\n{cp} BELUM LULUS. Penghambat: {rep['penghambat']}\nmanifest.json TIDAK ditulis.")
    return rep


def log_row(who: str, what: str, hasil: str = "disetujui", keputusan: str = "via notebook") -> None:
    line = f"| {dt.date.today().isoformat()} | {who} | {what} | {hasil} | {keputusan} |\n"
    if LOG_PATH.exists():
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line)


def approve_flag(cp: str, name: str, by: str, what: str) -> None:
    d = P(load_merged(BASE_PATH), cp.lower())
    d.mkdir(parents=True, exist_ok=True)
    (d / name).write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, f"{what} ({cp})")
    print("ditulis", d / name)


def approve_review(cp: str, by: str) -> None:
    approve_flag(cp, "REVIEWED", by, "Tinjauan orang kedua")


def approve_decisions(ids: list, by: str) -> None:
    text = DEC_PATH.read_text(encoding="utf-8")
    for d in ids:
        pat = re.compile(rf"(^{d}:\n  status: )menunggu_konfirmasi", re.M)
        if not pat.search(text):
            print(f"{d}: tidak berstatus menunggu_konfirmasi (dilewati)")
            continue
        text = pat.sub(rf"\1diputuskan\n  dikonfirmasi_oleh: {by}\n  tanggal_konfirmasi: {dt.date.today().isoformat()}", text)
        log_row(by, f"Konfirmasi keputusan {d}")
        print(f"{d}: diputuskan oleh {by}")
    DEC_PATH.write_text(text, encoding="utf-8")


def approve_roi(by: str) -> None:
    t = BASE_PATH.read_text(encoding="utf-8")
    t2 = re.sub(r"(  roi:.*?\n    locked: )false", r"\1true", t, count=1, flags=re.S)
    if t2 == t and "locked: true" not in t:
        raise SystemExit("preprocess.roi.locked tidak ditemukan di base.yaml")
    BASE_PATH.write_text(t2, encoding="utf-8")
    d = P(load_merged(BASE_PATH), "cp2")
    d.mkdir(parents=True, exist_ok=True)
    (d / "ROI_APPROVED").write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, "Parameter Frangi dikunci setelah pemeriksaan visual 50 citra (FR-03)")
    print("preprocess.roi.locked=true; ditulis", d / "ROI_APPROVED")

In [ ]:
# Konfirmasi keputusan (hanya setelah dibaca di configs/decisions.yaml) dan tinjauan orang kedua, lalu jalankan sel berikutnya.
DECIDER = None      # contoh: 'Wahyu'
if DECIDER:
    approve_decisions(['D1', 'D2', 'D3', 'D4'], DECIDER)
REVIEWER = None
if REVIEWER:
    approve_review('CP0', REVIEWER)

In [ ]:
def finalize_cp0(cfg: dict) -> dict:
    out_dir = P(cfg, "cp0")
    items = []
    env_p = out_dir / "env_check.json"
    if env_p.exists():
        env = json.loads(env_p.read_text(encoding="utf-8"))
        items.append(_item("Uji lingkungan (impor, dataset, bobot Swin-T)", LULUS if env.get("lulus") else GAGAL, f"gagal={env.get('gagal')}"))
        gs = env.get("gpu_smoke", {"status": TIDAK_TERSEDIA, "detail": "env_check lama tanpa uji GPU"})
        items.append(_item("Uji kecil di GPU", gs["status"], gs["detail"]))
        dev = env.get("device", {})
        items.append(_item("Perangkat tercatat (D1)", LULUS, f"GPU={dev.get('gpu_names')}, mem={dev.get('gpu_memory_gib')} GiB, "
                           f"CPU={dev.get('cpu_count')}, thread torch={dev.get('torch_threads')}"))
        items.append(_item("Hash commit dan versi pustaka tercatat (pengganti digest citra Docker)",
                           LULUS if env.get("git_commit") not in (None, "", "tanpa-commit") and env.get("pip_freeze_sha256") else GAGAL,
                           f"commit={env.get('git_commit')}, pip_freeze_sha256={str(env.get('pip_freeze_sha256'))[:12]}"))
    else:
        items.append(_item("Uji lingkungan", GAGAL, "env_check.json tidak ada"))
    for name, nama in (("benchmark.json", "Waktu Swin-T tercatat (tensor acak, ekstrapolasi)"),
                       ("benchmark_gan.json", "Waktu dan memori GAN tercatat (tensor acak, termasuk R1)")):
        p = out_dir / name
        if p.exists():
            bm = json.loads(p.read_text(encoding="utf-8"))
            if "mean_step_seconds" in bm:
                det = (f"batch {bm.get('batch_size')}: {bm['mean_step_seconds']:.2f} dtk/langkah, ~{bm['estimated_epoch_minutes']:.1f} mnt/epoch, "
                       f"~{bm['estimated_30_epoch_hours_per_job']:.1f} jam/job 30 epoch, device={bm.get('device')}, gpu={bm.get('gpu')}")
                ok = bm["mean_step_seconds"] > 0
            else:
                est = bm.get("estimasi") or {}
                det = (f"micro_batch S5/S6={est.get('micro_batch_S5')}/{est.get('micro_batch_S6')}, "
                       f"~{est.get('jam_total_GAN', float('nan')):.1f} jam GAN total (total_kimg={est.get('total_kimg')}), gpu={bm.get('gpu')}")
                ok = bool(bm.get("sapuan")) and bool(est)
            items.append(_item(nama, LULUS if ok else GAGAL, det))
        else:
            items.append(_item(nama, GAGAL, f"{name} tidak ada"))
    for d in ("D1", "D2", "D3", "D4"):
        items.append(_decision_item(d))
    items.append(_review_item(out_dir))
    return _finish("CP0", out_dir, items, [out_dir / "env_check.json", out_dir / "benchmark.json", out_dir / "benchmark_gan.json",
                                          DEC_PATH, BASE_PATH], {"git_commit": GIT_COMMIT})


report = finalize_cp0(cfg)
assert report['checkpoint'] == 'CP0'